In [ ]:
!pip install thefuzz python-Levenshtein rioxarray shapely

In [ ]:
import os
import glob
import pandas as pd
import os
import glob
import gc
import psutil
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray
import warnings
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg') # Keep Agg backend active for memory safety
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import make_interp_spline
from IPython.display import Image, display

In [ ]:
print("================ POOLING ALL COUNTRY DATASETS ================")

processed_files = glob.glob('/content/Processed_Country_Data/*_DHS_Spatial_Merged.csv')
print(f"Found {len(processed_files)} country datasets to pool.")

pooled_list = []
for f in processed_files:
    country_name = os.path.basename(f).replace('_DHS_Spatial_Merged.csv', '')
    df_c = pd.read_csv(f)
    df_c['country_name'] = country_name

    # Create a unique cluster ID across countries to avoid cluster ID collisions (e.g., cluster 1 in Benin vs cluster 1 in Ghana)
    df_c['global_cluster_id'] = df_c['country_name'].astype(str) + '_' + df_c['v001'].astype(str)

    # Create a unique province ID across countries for province fixed effects
    df_c['global_province_id'] = df_c['country_name'].astype(str) + '_' + df_c['v024'].astype(str)

    pooled_list.append(df_c)

df_pooled_master = pd.concat(pooled_list, ignore_index=True)

# Save the pooled master file locally
out_pooled_path = '/content/ALL_AFRICA_DHS_Spatial_Merged.csv'
df_pooled_master.to_csv(out_pooled_path, index=False)

print(f"\nSuccessfully created pooled dataset!")
print(f"Total Pooled Victims: {len(df_pooled_master)}")
print(f"Countries included: {df_pooled_master['country_name'].unique().tolist()}")

In [ ]:
def check_ram(step_name):
    ram = psutil.virtual_memory()
    print(f"[RAM] {step_name}: {ram.percent}% used ({ram.used / 1024**3:.2f} GB)")

# Suppress runtime warnings and statsmodels warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', message="cov_type not fully supported with freq_weights")

# 1. Load Pooled Data
check_ram("Before loading main CSV")
df_pooled = pd.read_csv('/content/ALL_AFRICA_DHS_Spatial_Merged.csv')
df_pooled['interview_date'] = pd.to_datetime(df_pooled['interview_date'])
check_ram("After loading main CSV")

# Ensure 'informal' outcome is explicitly defined if missing
if 'informal' not in df_pooled.columns and 'sought_any' in df_pooled.columns and 'formal' in df_pooled.columns:
    df_pooled['informal'] = ((df_pooled['sought_any'] == 1) & (df_pooled['formal'] == 0)).astype(int)

TARGET_MAP = {
    'informal': 'Sought Informal Help',
    'sought_any': 'Disclosed Violence (Any)',
    'formal': 'Sought Formal Help'
}

# --- POOLED MODEL SPECIFICATION ---
# REMOVED C(v000) to eliminate perfect multicollinearity with C(country_name)
demographics_cols = ['v012', 'v025', 'v106', 'v190', 'country_name', 'global_cluster_id']
demographics_formula = "v012 + C(v025) + C(v106) + C(v190) + C(country_name)"

scaler = StandardScaler()
df_pooled['weight'] = df_pooled['d005'] / 1e6

# ---------------------------------------------------------
# A. POOLED PRECIPITATION EXTRACTION & MODELING
# ---------------------------------------------------------
print("\n================ RUNNING POOLED PRECIPITATION ANALYSIS ================")

# 4% threshold buckets (5 buckets total)
threshold_bands = [(80, 84), (84, 88), (88, 92), (92, 96), (96, 100)]
precip_dirs = glob.glob('/content/drive/MyDrive/ERA5_Precip_*')

all_precip_metrics = []
for p_dir in precip_dirs:
    c_name = os.path.basename(p_dir).replace('ERA5_Precip_', '')
    df_c = df_pooled[df_pooled['country_name'] == c_name].copy()

    if len(df_c) == 0: continue

    tif_files = sorted(glob.glob(os.path.join(p_dir, "*.tif")))
    if not tif_files: continue

    print(f"\n  [DATA EXTRACT] Processing Precipitation for: {c_name}...")

    def load_tiff(f):
        da = rioxarray.open_rasterio(f, chunks={'x': 'auto', 'y': 'auto'})
        names = da.attrs.get('long_name') or da.long_name
        t = pd.to_datetime([b.split('_')[0] for b in names], format='%Y%m%d')
        return da.assign_coords(band=t).rename({'band': 'time', 'x': 'lon', 'y': 'lat'})

    ds_cube = xr.concat([load_tiff(f) for f in tif_files], dim='time')

    # 1. Compute Percentiles (Full Resolution)
    baseline_mask = ds_cube.time <= pd.to_datetime('2010-12-31')
    sampled_baseline = ds_cube.isel(time=baseline_mask)

    b_vals = sampled_baseline.values.flatten()
    b_vals = b_vals[b_vals > 0.0001]

    if len(b_vals) == 0:
        print(f"  --> WARNING: No valid baseline precipitation data found for {c_name}. Skipping...")
        continue

    u_pcts = sorted(list(set([p for band in threshold_bands for p in band])))
    pct_thresh = dict(zip(u_pcts, np.nanpercentile(b_vals, u_pcts)))

    del b_vals, sampled_baseline
    gc.collect()

    # 2. Extract Data to Clusters
    clusters = df_c[['global_cluster_id', 'v001', 'LONGNUM', 'LATNUM']].drop_duplicates()
    lons_da = xr.DataArray(clusters['LONGNUM'].values, dims='cluster')
    lats_da = xr.DataArray(clusters['LATNUM'].values, dims='cluster')

    cluster_cube = ds_cube.sel(lon=lons_da, lat=lats_da, method='nearest').compute()
    cluster_cube['cluster_id'] = ('cluster', clusters['global_cluster_id'].values)
    cluster_cube = cluster_cube.swap_dims({'cluster': 'cluster_id'})

    ds_cube.close()
    del ds_cube
    gc.collect()

    # 3. Process time windows
    unique_ctx = df_c[['global_cluster_id', 'interview_date']].drop_duplicates()
    for _, r in unique_ctx.iterrows():
        gc_id, end_w = r['global_cluster_id'], r['interview_date']
        start_w = end_w - pd.DateOffset(years=1)
        w_vals = cluster_cube.sel(cluster_id=gc_id, time=slice(start_w, end_w)).values

        if len(w_vals) == 0: continue
        m = {'global_cluster_id': gc_id, 'interview_date': end_w}
        for low, upp in threshold_bands:
            m[f"exceed_{low}_{upp}"] = np.sum((w_vals >= pct_thresh[low]) & (w_vals <= pct_thresh[upp]))
        all_precip_metrics.append(m)

    del cluster_cube
    gc.collect()

print(f"\n[DIAGNOSTIC] Merge Phase: Extracted data for {len(all_precip_metrics)} unique cluster-interview pairs.")
df_precip_pooled = df_pooled.merge(pd.DataFrame(all_precip_metrics), on=['global_cluster_id', 'interview_date'], how='inner')

# ---------------------------------------------------------
# B. DIAGNOSTIC GLM MODELING
# ---------------------------------------------------------
print("\n================ RUNNING DIAGNOSTIC GLMs ================")
results_precip_pooled = []

for lower, upper in threshold_bands:
    col = f"exceed_{lower}_{upper}"
    if col not in df_precip_pooled.columns: continue
    df_precip_pooled[f'z_{col}'] = scaler.fit_transform(df_precip_pooled[[col]])

    for v_col, v_title in TARGET_MAP.items():
        print(f"\n[MODEL] Band {lower}-{upper}% | Outcome: {v_title}")

        df_clean = df_precip_pooled.dropna(subset=[v_col, f'z_{col}'] + demographics_cols + ['weight']).copy()

        if len(df_clean) < 100 or df_clean[v_col].nunique() < 2:
            print("  --> SKIPPED: Sample size too small (< 100) or zero overall variance.")
            continue

        # Filter zero-variance countries to prevent singular matrix errors
        country_var = df_clean.groupby('country_name')[v_col].nunique()
        valid_countries = country_var[country_var > 1].index
        df_clean = df_clean[df_clean['country_name'].isin(valid_countries)].copy()

        print(f"  --> Data retained after filtering zero-variance countries: {len(df_clean)} rows")
        print(f"  --> Unique valid countries in this regression: {len(valid_countries)}")

        for cat_col in ['v025', 'v106', 'v190', 'country_name']:
            df_clean[cat_col] = df_clean[cat_col].astype(str)

        formula = f"{v_col} ~ z_{col} + {demographics_formula}"

        try:
            group_codes = pd.factorize(df_clean['global_cluster_id'])[0]
            model = smf.glm(formula=formula, data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['weight'])

            # Primary fit: Clustered Standard Errors
            res = model.fit(cov_type='cluster', cov_kwds={'groups': group_codes})
            bse_val = res.bse.get(f'z_{col}', np.nan)

            # Fallback fit: Robust HC1 SEs if cluster SE computation returned NaN
            if np.isnan(bse_val):
                print("  --> WARNING: Cluster SE returned NaN. Falling back to robust HC1 standard errors...")
                res = model.fit(cov_type='HC1')
                bse_val = res.bse[f'z_{col}']

            coef_val = res.params[f'z_{col}']
            pval_val = res.pvalues[f'z_{col}']

            results_precip_pooled.append({
                'Outcome': v_title,
                'Lower_Bound': lower,
                'Coef': coef_val,
                'CI_Lower': coef_val - (1.96 * bse_val),
                'CI_Upper': coef_val + (1.96 * bse_val),
                'Significant': pval_val < 0.05
            })
            print("  --> SUCCESS: Model converged with valid SEs.")
        except Exception as e:
            print(f"  --> FAILED: statsmodels error: {e}")

df_res_p_pooled = pd.DataFrame(results_precip_pooled)

# ---------------------------------------------------------
# C. PLOTTING & INLINE DISPLAY
# ---------------------------------------------------------
if not df_res_p_pooled.empty:
    print("\n[PLOT] Generating final visualization...")
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

    fig.suptitle('POOLED AFRICA SAMPLE: Extreme Rainfall & Help-Seeking Sensitivity\n(Country FE | Clustered SEs by Community)', fontsize=16, weight='bold')

    for idx, (v_col, v_title) in enumerate(TARGET_MAP.items()):
        ax = axes[idx]
        v_data = df_res_p_pooled[df_res_p_pooled['Outcome'] == v_title].sort_values('Lower_Bound').copy()

        if v_data.empty:
            ax.set_title(f"{v_title}\n(No valid models)")
            continue

        v_data[['Pct_Change', 'CI_L', 'CI_U']] = (np.exp(v_data[['Coef', 'CI_Lower', 'CI_Upper']]) - 1) * 100

        # Filter out NaN/Inf values before plotting
        v_data = v_data[np.isfinite(v_data['Pct_Change']) & np.isfinite(v_data['CI_L']) & np.isfinite(v_data['CI_U'])].copy()

        if v_data.empty:
            ax.set_title(f"{v_title}\n(Non-finite values)")
            continue

        x = v_data['Lower_Bound']
        y = v_data['Pct_Change']

        ax.axhline(0, color='black', linestyle='--')
        ax.fill_between(x, v_data['CI_L'], v_data['CI_U'], color='#1f77b4', alpha=0.2)

        # Safe spline fitting with direct line fallback
        if len(v_data) >= 4:
            try:
                x_smooth = np.linspace(x.min(), x.max(), 300)
                y_smooth = make_interp_spline(x, y, k=3)(x_smooth)
                ax.plot(x_smooth, y_smooth, color='#1f77b4', linewidth=2)
            except Exception:
                ax.plot(x, y, color='#1f77b4', linewidth=2, marker='o')
        else:
            ax.plot(x, y, color='#1f77b4', linewidth=2, marker='o')

        ax.scatter(x, y, color=['#d62728' if sig else '#1f77b4' for sig in v_data['Significant']], s=60, edgecolor='white', zorder=5)
        ax.set_title(v_title, weight='bold')
        ax.set_xlabel('Rainfall Band Lower Bound (%)')
        if idx == 0: ax.set_ylabel('% Change in Odds of Help-Seeking')

    plt.tight_layout()
    out_img_path = '/content/POOLED_AFRICA_precipitation_sensitivity.png'
    plt.savefig(out_img_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    print("\nPooled Analysis Complete! Rendering inline plot below:")
    display(Image(filename=out_img_path))
else:
    print("\n[CRITICAL] No models successfully converged. Plotting skipped.")